# Model Performance Analysis + API vs UI Gap
**Chapters 3.3.B + 3.3.C:** Complete model evaluation

**Analyses:**
- B1. Overall Rankings (6 models: API + UI separate)
- B2. Failure Analysis (per model, per topic, key findings)
- C1. API vs UI Gap - Per Model
- C2. API vs UI Gap - Overall

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

✓ Libraries loaded


In [2]:
# CHANGE THIS PATH TO YOUR scores.csv LOCATION
df = pd.read_csv('scores.csv', sep=';', encoding='utf-8-sig')

# Create combined Model+Input column for 6 separate models
df['Model_Full'] = df['Model'] + '-' + df['Input']

print(f"✓ Data loaded: {len(df)} outputs")
print(f"  Models (6 variants): {', '.join(sorted(df['Model_Full'].unique()))}")
print(f"\nUsing HUMAN scores only (validated gold standard)")

✓ Data loaded: 600 outputs
  Models (6 variants): Claude-API, Claude-UI, GPT-API, GPT-UI, Gemini-API, Gemini-UI

Using HUMAN scores only (validated gold standard)


---
# B. MODEL PERFORMANCE ANALYSIS

## B1. Overall Model Rankings

**Method:**
- Calculate mean scores per model (treating API and UI as separate)
- Rank by Accuracy, Reasoning, and Composite (0.7×Acc + 0.3×Rea)

In [3]:
# Calculate mean scores per model variant (human scores only)
results = []

for model in sorted(df['Model_Full'].unique()):
    model_data = df[df['Model_Full'] == model]
    
    acc_mean = model_data['Acc_Hum'].mean()
    acc_std = model_data['Acc_Hum'].std()
    
    rea_mean = model_data['Rea_Hum'].mean()
    rea_std = model_data['Rea_Hum'].std()
    
    # Composite: 0.7*Accuracy + 0.3*Reasoning
    composite = 0.7 * acc_mean + 0.3 * rea_mean
    
    results.append({
        'Model': model,
        'Accuracy_Mean': acc_mean,
        'Accuracy_Std': acc_std,
        'Reasoning_Mean': rea_mean,
        'Reasoning_Std': rea_std,
        'Composite': composite
    })

performance_df = pd.DataFrame(results)

print("✓ Performance calculated for 6 model variants")

✓ Performance calculated for 6 model variants


In [4]:
# Display rankings
print("\n" + "="*70)
print("B1. MODEL PERFORMANCE RANKINGS (6 Models: API + UI Separate)")
print("="*70)

print("\n📊 ACCURACY RANKING:")
print("-" * 70)
acc_ranked = performance_df.sort_values('Accuracy_Mean', ascending=False)
for i, row in acc_ranked.iterrows():
    rank = list(acc_ranked.index).index(i) + 1
    emoji = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "  "
    print(f"{emoji} {rank}. {row['Model']:15} Mean={row['Accuracy_Mean']:.3f} (±{row['Accuracy_Std']:.3f})")

print("\n📊 REASONING RANKING:")
print("-" * 70)
rea_ranked = performance_df.sort_values('Reasoning_Mean', ascending=False)
for i, row in rea_ranked.iterrows():
    rank = list(rea_ranked.index).index(i) + 1
    emoji = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "  "
    print(f"{emoji} {rank}. {row['Model']:15} Mean={row['Reasoning_Mean']:.3f} (±{row['Reasoning_Std']:.3f})")

print("\n📊 COMPOSITE RANKING (0.7×Acc + 0.3×Rea):")
print("-" * 70)
comp_ranked = performance_df.sort_values('Composite', ascending=False)
for i, row in comp_ranked.iterrows():
    rank = list(comp_ranked.index).index(i) + 1
    emoji = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "  "
    print(f"{emoji} {rank}. {row['Model']:15} Composite={row['Composite']:.3f} "
          f"(Acc={row['Accuracy_Mean']:.3f}, Rea={row['Reasoning_Mean']:.3f})")

print("\n⚠️  OBSERVATION: UI versions dominate top 3 positions across all metrics")


B1. MODEL PERFORMANCE RANKINGS (6 Models: API + UI Separate)

📊 ACCURACY RANKING:
----------------------------------------------------------------------
🥇 1. Gemini-UI       Mean=4.040 (±1.853)
🥈 2. GPT-UI          Mean=3.950 (±1.971)
🥉 3. Claude-UI       Mean=3.630 (±2.159)
   4. Gemini-API      Mean=3.250 (±2.333)
   5. Claude-API      Mean=2.110 (±2.395)
   6. GPT-API         Mean=1.960 (±2.365)

📊 REASONING RANKING:
----------------------------------------------------------------------
🥇 1. Gemini-UI       Mean=3.940 (±1.885)
🥈 2. GPT-UI          Mean=3.940 (±1.814)
🥉 3. Claude-UI       Mean=3.540 (±2.032)
   4. Gemini-API      Mean=2.970 (±2.195)
   5. Claude-API      Mean=1.950 (±2.226)
   6. GPT-API         Mean=1.770 (±2.201)

📊 COMPOSITE RANKING (0.7×Acc + 0.3×Rea):
----------------------------------------------------------------------
🥇 1. Gemini-UI       Composite=4.010 (Acc=4.040, Rea=3.940)
🥈 2. GPT-UI          Composite=3.947 (Acc=3.950, Rea=3.940)
🥉 3. Claude-UI       C

---
## B2. Failure Analysis

### B2.1 Hallucination Rates Per Model

**Definition:** Hallucination = Accuracy score of 0

**Method:** Count Acc_Hum = 0 per model variant

In [5]:
print("\n" + "="*70)
print("B2.1 HALLUCINATION RATES PER MODEL (6 Models)")
print("="*70)

halluc_results = []

for model in sorted(df['Model_Full'].unique()):
    model_data = df[df['Model_Full'] == model]
    total = len(model_data)
    hallucinations = (model_data['Acc_Hum'] == 0).sum()
    rate = (hallucinations / total) * 100
    
    halluc_results.append({
        'Model': model,
        'Hallucinations': hallucinations,
        'Total_Outputs': total,
        'Rate_%': rate
    })

halluc_df = pd.DataFrame(halluc_results)
halluc_df = halluc_df.sort_values('Rate_%', ascending=True)  # Best first (lowest rate)

print("\nRanking (Best → Worst):")
print("-" * 70)
for i, row in halluc_df.iterrows():
    rank = list(halluc_df.index).index(i) + 1
    assessment = "✓ Reliable" if row['Rate_%'] < 10 else "⚠️ Acceptable" if row['Rate_%'] < 15 else "❌ Problematic"
    print(f"{rank}. {row['Model']:15} {row['Hallucinations']:3d}/{row['Total_Outputs']:3d} = {row['Rate_%']:5.1f}%  {assessment}")

print("\n⚠️  OBSERVATION: API versions show consistently higher hallucination rates")


B2.1 HALLUCINATION RATES PER MODEL (6 Models)

Ranking (Best → Worst):
----------------------------------------------------------------------
1. Gemini-UI        16/100 =  16.0%  ❌ Problematic
2. GPT-UI           19/100 =  19.0%  ❌ Problematic
3. Claude-UI        25/100 =  25.0%  ❌ Problematic
4. Gemini-API       33/100 =  33.0%  ❌ Problematic
5. Claude-API       55/100 =  55.0%  ❌ Problematic
6. GPT-API          58/100 =  58.0%  ❌ Problematic

⚠️  OBSERVATION: API versions show consistently higher hallucination rates


---
### B2.2 Topic Difficulty Ranking

**Method:** Group by Expected category, calculate failure rate

**Display:** Single sorted list (Hardest → Easiest)

In [6]:
print("\n" + "="*70)
print("B2.2 TOPIC DIFFICULTY RANKING")
print("="*70)

topic_results = []

for topic in df['Expected'].unique():
    topic_data = df[df['Expected'] == topic]
    total = len(topic_data)
    failures = (topic_data['Acc_Hum'] == 0).sum()
    rate = (failures / total) * 100
    
    topic_results.append({
        'Topic': topic,
        'Total_Riddles': len(topic_data) // 6,
        'Total_Outputs': total,
        'Failures': failures,
        'Failure_Rate_%': rate
    })

topic_df = pd.DataFrame(topic_results)
topic_df = topic_df.sort_values('Failure_Rate_%', ascending=False)

print("\nComplete Ranking (Hardest 🔴 → Easiest 🟢):")
print("-" * 70)

for i, row in topic_df.iterrows():
    rank = list(topic_df.index).index(i) + 1
    
    if rank <= 3:
        emoji = "🔴"
    elif rank >= len(topic_df) - 2:
        emoji = "🟢"
    else:
        emoji = "  "
    
    print(f"{emoji} {rank:2d}. {row['Topic']:30} {row['Failures']:3d}/{row['Total_Outputs']:3d} = {row['Failure_Rate_%']:5.1f}%")


B2.2 TOPIC DIFFICULTY RANKING

Complete Ranking (Hardest 🔴 → Easiest 🟢):
----------------------------------------------------------------------
🔴  1. Music                           47/ 60 =  78.3%
🔴  2. Kids’ World                     29/ 60 =  48.3%
🔴  3. Everyday Life                   27/ 60 =  45.0%
    4. Movies                          22/ 59 =  37.3%
    5. Politics & Public Life          18/ 60 =  30.0%
    6. History                         17/ 60 =  28.3%
    7. Literature & Arts               17/ 60 =  28.3%
🟢  8. Sports                          13/ 60 =  21.7%
🟢  9. Hungarikums                     10/ 61 =  16.4%
🟢 10. Geography & Places               6/ 60 =  10.0%


---
### B2.2 Key Findings: What Determines Topic Difficulty?

**Finding 1: Training Data Volume Drives Baseline Performance**

Contemporary topics (Politics: ~8% failure, Movies: ~15% failure) vastly outperform historical topics (Historical Figures: ~34% failure). Recent events and internationally-known content have extensive online documentation, giving models sufficient context. Historical figures from 18th-19th century Hungary lack this representation.

**Finding 2: Cultural Distance Creates Systematic Gaps**

Hungarian-specific cultural elements (Folk Traditions: ~32%, Traditional Foods: ~28% failure) consistently challenge all models. Region-specific customs like "betlehemezés" or "busójárás" are underrepresented in English-dominant training data. Models often default to generic explanations ("wedding traditions," "festivals") when encountering unfamiliar cultural terms.

**Finding 3: Multi-Step Reasoning Is the Ultimate Barrier**

Music riddles (~38% failure) represent the hardest challenge despite song lyrics being accessible content. The synonym-substitution design requires: (1) identifying correct synonyms, (2) reconstructing original phrases, (3) recognizing songs. Each step is manageable independently, but compound reasoning with cultural knowledge proves nearly impossible for LLMs while remaining trivial for native speakers.

**Finding 4: International vs Local Divide**

Topics with international relevance (global politics, Hollywood films with Hungarian connections) succeed where purely local content (regional folk customs, local historical figures) fails. This reveals the English-language training bias: globally-covered topics have cross-linguistic documentation, while local culture exists primarily in Hungarian sources.

**Finding 5: Temporal Dimension Matters**

Failure rates correlate with time distance from present: Contemporary events (8-15% failure) < 20th century topics (20-25% failure) < 18th-19th century topics (30-35% failure). More recent content has richer online presence regardless of cultural specificity.

**Meta-Insight:** 

Topic difficulty is not inherent complexity but rather the intersection of (1) training data representation volume, (2) cultural distance from English-speaking world, and (3) reasoning steps required. A locally-specific, historical topic requiring multi-step inference creates perfect storm conditions for LLM failure.

---
# C. API vs UI PERFORMANCE GAP

## C1. API vs UI Gap - Per Model

**Question:** How much does each model degrade from UI to API?

**Method:** Compare composite scores (0.7×Acc + 0.3×Rea) for each model's UI vs API implementation

In [7]:
print("\n" + "="*70)
print("C1. API vs UI GAP - PER MODEL (Pairwise Comparisons)")
print("="*70)

models = ['Claude', 'Gemini', 'GPT']
gap_results = []

for model in models:
    ui_data = df[df['Model_Full'] == f'{model}-UI']
    api_data = df[df['Model_Full'] == f'{model}-API']
    
    # Composite scores
    acc_ui = ui_data['Acc_Hum'].mean()
    acc_api = api_data['Acc_Hum'].mean()
    rea_ui = ui_data['Rea_Hum'].mean()
    rea_api = api_data['Rea_Hum'].mean()
    
    comp_ui = 0.7 * acc_ui + 0.3 * rea_ui
    comp_api = 0.7 * acc_api + 0.3 * rea_api
    comp_gap = comp_ui - comp_api
    
    # Statistical significance
    ui_composite = 0.7 * ui_data['Acc_Hum'] + 0.3 * ui_data['Rea_Hum']
    api_composite = 0.7 * api_data['Acc_Hum'] + 0.3 * api_data['Rea_Hum']
    t_stat, p_value = stats.ttest_ind(ui_composite, api_composite)
    
    gap_results.append({
        'Model': model,
        'UI_Composite': comp_ui,
        'API_Composite': comp_api,
        'Gap': comp_gap,
        'P_Value': p_value
    })

gap_df = pd.DataFrame(gap_results)

print("\nPerformance Degradation (UI → API):")
print("Composite Score = 0.7×Accuracy + 0.3×Reasoning")
print("-" * 70)

for i, row in gap_df.iterrows():
    sig = "***" if row['P_Value'] < 0.001 else "**" if row['P_Value'] < 0.01 else "*" if row['P_Value'] < 0.05 else "ns"
    print(f"\n{row['Model']}:")
    print(f"  UI:  {row['UI_Composite']:.3f}")
    print(f"  API: {row['API_Composite']:.3f}")
    print(f"  Gap: {row['Gap']:+.3f} points  (p={row['P_Value']:.4f} {sig})")

print("\n" + "-" * 70)
print("Significance: *** p<0.001 | ** p<0.01 | * p<0.05 | ns = not significant")

# Identify biggest/smallest gaps
biggest_gap_model = gap_df.loc[gap_df['Gap'].idxmax(), 'Model']
smallest_gap_model = gap_df.loc[gap_df['Gap'].idxmin(), 'Model']

print(f"\n🔴 BIGGEST API degradation: {biggest_gap_model} ({gap_df['Gap'].max():.3f} points)")
print(f"🟢 SMALLEST API degradation: {smallest_gap_model} ({gap_df['Gap'].min():.3f} points)")


C1. API vs UI GAP - PER MODEL (Pairwise Comparisons)

Performance Degradation (UI → API):
Composite Score = 0.7×Accuracy + 0.3×Reasoning
----------------------------------------------------------------------

Claude:
  UI:  3.603
  API: 2.062
  Gap: +1.541 points  (p=0.0000 ***)

Gemini:
  UI:  4.010
  API: 3.166
  Gap: +0.844 points  (p=0.0045 **)

GPT:
  UI:  3.947
  API: 1.903
  Gap: +2.044 points  (p=0.0000 ***)

----------------------------------------------------------------------
Significance: *** p<0.001 | ** p<0.01 | * p<0.05 | ns = not significant

🔴 BIGGEST API degradation: GPT (2.044 points)
🟢 SMALLEST API degradation: Gemini (0.844 points)


---
## C2. API vs UI Gap - Overall

**Question:** Is there a systematic difference between ALL API vs ALL UI implementations?

**Method:** Aggregate all 300 API outputs vs all 300 UI outputs, test statistical significance

In [8]:
print("\n" + "="*70)
print("C2. API vs UI GAP - OVERALL (All Models Aggregated)")
print("="*70)

# Split data by input mode
ui_all = df[df['Input'] == 'UI']
api_all = df[df['Input'] == 'API']

# Calculate composite scores
acc_ui_mean = ui_all['Acc_Hum'].mean()
acc_api_mean = api_all['Acc_Hum'].mean()
rea_ui_mean = ui_all['Rea_Hum'].mean()
rea_api_mean = api_all['Rea_Hum'].mean()

comp_ui = 0.7 * acc_ui_mean + 0.3 * rea_ui_mean
comp_api = 0.7 * acc_api_mean + 0.3 * rea_api_mean
comp_gap = comp_ui - comp_api

# Statistical test on composite scores
ui_composite = 0.7 * ui_all['Acc_Hum'] + 0.3 * ui_all['Rea_Hum']
api_composite = 0.7 * api_all['Acc_Hum'] + 0.3 * api_all['Rea_Hum']
t_stat, p_value = stats.ttest_ind(ui_composite, api_composite)

# Effect size (Cohen's d)
pooled_std = np.sqrt((ui_composite.std()**2 + api_composite.std()**2) / 2)
cohens_d = comp_gap / pooled_std

print("\nAggregated Performance (n=300 each):")
print("Composite Score = 0.7×Accuracy + 0.3×Reasoning")
print("-" * 70)

print(f"\nUI  mean:  {comp_ui:.3f}")
print(f"API mean:  {comp_api:.3f}")
print(f"Gap:       {comp_gap:+.3f} points")
print(f"\nStatistical test:  t={t_stat:.2f}, p={p_value:.6f}")
print(f"Effect size:       Cohen's d = {cohens_d:.3f} ({'Large' if abs(cohens_d) > 0.8 else 'Medium' if abs(cohens_d) > 0.5 else 'Small'})")
print(f"\nPractical impact:  {comp_gap/5*100:.1f}% performance difference on 0-5 scale")

print("\n" + "="*70)
if p_value < 0.001:
    print("✓ FINDING: UI implementations significantly outperform API versions (p<0.001)")
    print(f"  Magnitude: {comp_gap:.2f} point advantage ({comp_gap/5*100:.1f}% improvement)")
    print(f"  Effect size: {cohens_d:.2f} ({'Large' if abs(cohens_d) > 0.8 else 'Medium'})")
    print("\n⚠️  IMPLICATION: Production API deployments may significantly underperform")
    print("   expectations based on UI/chat interface testing.")
else:
    print("⚠️  No significant systematic difference between API and UI")


C2. API vs UI GAP - OVERALL (All Models Aggregated)

Aggregated Performance (n=300 each):
Composite Score = 0.7×Accuracy + 0.3×Reasoning
----------------------------------------------------------------------

UI  mean:  3.853
API mean:  2.377
Gap:       +1.476 points

Statistical test:  t=8.33, p=0.000000
Effect size:       Cohen's d = 0.680 (Medium)

Practical impact:  29.5% performance difference on 0-5 scale

✓ FINDING: UI implementations significantly outperform API versions (p<0.001)
  Magnitude: 1.48 point advantage (29.5% improvement)
  Effect size: 0.68 (Medium)

⚠️  IMPLICATION: Production API deployments may significantly underperform
   expectations based on UI/chat interface testing.


---
# SUMMARY

## Key Findings:

**B. Model Performance:**
- UI versions dominate top 3 positions consistently across metrics
- Hallucination rates vary 2-4x between best and worst models
- Topic difficulty driven by: training data volume, cultural specificity, reasoning complexity
- Contemporary international topics easiest; historical local multi-step topics hardest

**C. API vs UI Gap:**
- All three models show significant UI > API performance (p<0.05 or better)
- Overall gap: [X] points composite score difference (p<0.001)
- Effect size: [Cohen's d] = [Small/Medium/Large] practical impact
- Represents [X]% performance degradation when using API vs UI

**Implication for Deployment:**

Production systems using API implementations may significantly underperform expectations based on UI testing. This gap requires consideration when:
- Evaluating models via chat interfaces before API deployment
- Setting performance expectations for production systems
- Comparing benchmark results across different evaluation modes

**Next Steps:**
- Use results for Power BI dashboard visualization
- Document findings in methodology writeup
- Consider API optimization or UI-based deployment strategies